# DriveSense-VLM — 01: SFT Training

**GPU**: A100 80GB (required) | **Time**: ~3–5 h | **Cost**: ~50 CU

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **RECOVERY**: Rerun Cells 2–3, then rerun the training cell. Auto-resumes from checkpoint.

**Prerequisites**: `00_data_pipeline.ipynb` must have generated `sft_train.jsonl`.

In [6]:
# ══════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════
RUN_DEBUG_FIRST   = True   # Run 10-step smoke test before full training
INSTALL_FLASH_ATTN = False  # True: install flash-attn (takes 15+ min to compile)
                             # False: use built-in SDPA fallback (recommended)
# ══════════════════════════════════════════════════════════

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD (skip if already cloned — saves bandwidth)
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

Mounted at /content/drive
Cloning into '/content/DriveSense-VLM'...
remote: Enumerating objects: 288, done.
remote: Counting objects: 100% (288/288), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 288 (delta 129), reused 246 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (288/288), 428.14 KiB | 11.27 MiB/s, done.
Resolving deltas: 100% (129/129), done.
Working dir: /content/DriveSense-VLM
✓ Repo:  /content/DriveSense-VLM
✓ Drive: /content/drive/MyDrive/DriveSense-VLM


In [7]:
# Install training dependencies
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate datasets bitsandbytes wandb -q 2>&1 | tail -1

if INSTALL_FLASH_ATTN:
    print("Installing flash-attn (may take 15+ min)...")
    !pip install flash-attn --no-build-isolation -q 2>&1 | tail -3
    print("✓ flash-attn installed")
else:
    print("Flash-attn: skipped (INSTALL_FLASH_ATTN=False). SDPA fallback will be used.")

import torch
assert torch.cuda.is_available(), "No GPU! Set Runtime → A100"
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name} ({vram_gb:.0f} GB)")
if "A100" not in gpu_name:
    print(f"  WARNING: Expected A100 — got {gpu_name}. May OOM on <40 GB GPUs.")

import drivesense
print("✓ DriveSense loaded")

ipython 7.34.0 requires jedi>=0.16, which is not installed.
Flash-attn: skipped (INSTALL_FLASH_ATTN=False). SDPA fallback will be used.
✓ GPU: NVIDIA A100-SXM4-40GB (42 GB)
✓ DriveSense loaded


In [8]:
# Verify SFT data exists and image paths are populated
import os, json

SFT_DIR = f"{OUTPUTS_ROOT}/data/sft_ready"
print("SFT Data Verification:")
print("─" * 48)
for split in ("train", "val", "test"):
    path = f"{SFT_DIR}/sft_{split}.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            count = sum(1 for _ in f)
        with open(path) as f:
            ex = json.loads(f.readline())
        img = ""
        for msg in ex.get("messages", []):
            for item in (msg.get("content") if isinstance(msg.get("content"), list) else []):
                if item.get("type") == "image":
                    img = item.get("image", "")
        img_ok = "✓" if img and os.path.exists(img) else "✗ empty/missing"
        print(f"  ✓ sft_{split}.jsonl: {count} examples | image: {img_ok}")
    else:
        raise FileNotFoundError(
            f"sft_{split}.jsonl not found at {path}\n"
            "Run 00_data_pipeline.ipynb first."
        )

SFT Data Verification:
────────────────────────────────────────────────
  ✓ sft_train.jsonl: 2754 examples | image: ✓
  ✓ sft_val.jsonl: 344 examples | image: ✓
  ✓ sft_test.jsonl: 345 examples | image: ✓


In [9]:
import wandb
wandb.login()
print("✓ W&B configured — metrics will log to project: drivesense-vlm")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jayanth-kalyanam (jayanth-kalyanam-san-jose-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B configured — metrics will log to project: drivesense-vlm


## Debug Training (10 steps)

Creates `configs/training_debug.yaml` in the repo (not `/tmp/`) so multi-file
config resolution works. Always overwrites the debug config.

In [10]:
!pip install "torchao>=0.16.0" -q 2>&1 | tail -1

In [10]:
import os, yaml

os.chdir(REPO_ROOT)

with open("configs/training.yaml") as f:
    config = yaml.safe_load(f)

config["training"]["max_steps"]              = 10
config["training"]["num_epochs"]             = 1
config["training"]["logging_steps"]          = 1
config["training"]["save_strategy"]          = "no"
config["training"]["load_best_model_at_end"] = False
config["training"]["eval_strategy"]          = "no"

with open("configs/training_debug.yaml", "w") as f:
    yaml.dump(config, f)
print("✓ configs/training_debug.yaml written")

if RUN_DEBUG_FIRST:
    print("Running 10-step smoke test...")
    !python scripts/run_training.py --config configs/training_debug.yaml 2>&1
else:
    print("Skipping debug run (RUN_DEBUG_FIRST=False).")

✓ configs/training_debug.yaml written
Running 10-step smoke test...
02:10:36  WARNING   torchao  Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
02:10:37  INFO      numexpr.utils  NumExpr defaulting to 12 threads.
02:10:37  INFO      datasets  TensorFlow version 2.19.0 available.
02:10:37  INFO      datasets  JAX version 0.7.2 available.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jayanth-kalyanam (jayanth-kalyanam-san-jose-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run lqoyov02 (0.2s)
wandb: ⣽ setting up run lqoyov02 (0.2s)
wandb: ⣾ setting up run lqoyov02 (0.2s)
wandb: ⣷ setting up run lqoyov02 (0.2s)
wandb: ⣯ setting up run lqoyov02 (0.2s)
wandb: ⣟ setting up run lqoyov02 (0.7s)
wandb: ⡿ setting up run lqoyov02 (0.7s)
wandb: ⢿ setting 

## Full Training

Checkpoints saved to Drive every 250 steps. `--resume` auto-detects the latest checkpoint.

**RECOVERY**: Rerun Cells 2–3, then rerun this cell.

In [11]:
import shutil, os

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

# Delete debug run artifacts
for item in os.listdir(TRAIN_OUT) if os.path.exists(TRAIN_OUT) else []:
    full = f"{TRAIN_OUT}/{item}"
    if os.path.isdir(full):
        shutil.rmtree(full)
        print(f"Deleted {item}/")
    elif os.path.isfile(full):
        os.remove(full)
        print(f"Deleted {item}")

print("✓ Training output cleared — ready for fresh full training")

✓ Training output cleared — ready for fresh full training


In [26]:
os.chdir(REPO_ROOT)
!python scripts/run_training.py --config configs/training.yaml 2>&1

05:17:53  WARNING   torchao  Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
05:17:54  INFO      numexpr.utils  NumExpr defaulting to 12 threads.
05:17:54  INFO      datasets  TensorFlow version 2.19.0 available.
05:17:54  INFO      datasets  JAX version 0.7.2 available.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jayanth-kalyanam (jayanth-kalyanam-san-jose-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ setting up run 8oa0n6l7 (0.4s)
wandb: ⣷ setting up run 8oa0n6l7 (0.4s)
wandb: Tracking run with wandb version 0.26.0
wandb: Run data is saved locally in /content/DriveSense-VLM/wandb/run-20260428_051756-8oa0n6l7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bre

## Results

In [27]:
import os, json, glob

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

metrics_path = f"{TRAIN_OUT}/training_metrics.json"
if os.path.exists(metrics_path):
    m = json.loads(open(metrics_path).read())
    print("Training Metrics:")
    print("─" * 40)
    for k, v in m.items():
        print(f"  {k:<30}: {v}")
else:
    print(f"Metrics file not found at {metrics_path}")

best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

if best and os.path.exists(best):
    print(f"\n✓ Adapter saved: {best}")
    !ls -lh {best}
    print("\nProceed to 02_optimization.ipynb")
else:
    print("⚠ No checkpoint found — training may still be running or failed.")

Training Metrics:
────────────────────────────────────────
  train_loss                    : 11.205619172553796
  eval_loss                     : 11.165140151977539
  epochs_trained                : 4.0
  best_checkpoint               : outputs/training
  lora_adapter_dir              : outputs/training/lora_adapter

✓ Adapter saved: /content/drive/MyDrive/DriveSense-VLM/outputs/training/checkpoint-692
total 406M
-rw------- 1 root root 1.1K Apr 28 09:12 adapter_config.json
-rw------- 1 root root 136M Apr 28 09:12 adapter_model.safetensors
-rw------- 1 root root 271M Apr 28 09:12 optimizer.pt
-rw------- 1 root root 5.1K Apr 28 09:12 README.md
-rw------- 1 root root  15K Apr 28 09:12 rng_state.pth
-rw------- 1 root root 1.5K Apr 28 09:12 scheduler.pt
-rw------- 1 root root  14K Apr 28 09:12 trainer_state.json
-rw------- 1 root root 5.1K Apr 28 09:12 training_args.bin

Proceed to 02_optimization.ipynb
